[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/35_vit_patch.ipynb)

# 🟡 Medium: ViT Patch Embedding

*Attention & Transformers*
Implement the **patch embedding** stem of a Vision Transformer as an
`nnx.Module`: cut an image into non-overlapping square patches, flatten each
one, and linearly project it to a token vector.

$$x \in \mathbb{R}^{B \times H \times W \times C}
\;\longrightarrow\;
z \in \mathbb{R}^{B \times N \times E},
\qquad N = \frac{H}{P}\cdot\frac{W}{P}$$

$$z_n = \mathrm{flatten}(\text{patch}_n)\,W + b,
\qquad W \in \mathbb{R}^{(P^2 C) \times E}$$

### Rules
- Subclass `nnx.Module`; signature
  `PatchEmbedding(img_size, patch_size, in_channels, embed_dim, *, rngs)`
- Input is **NHWC** — `(B, H, W, C)` — the JAX/XLA channel-last convention,
  not PyTorch's NCHW
- Output `(B, N, embed_dim)` with patches in **row-major grid order**: patch
  index `n = row * (W // P) + col`
- Within a patch, flatten in `(patch_row, patch_col, channel)` order
- Expose `self.num_patches`
- Do the reshaping yourself — no `jax.lax.conv_general_dilated`, no
  `einops.rearrange`
- One projection matrix of shape `(P*P*C, embed_dim)` plus a bias, both
  `nnx.Param`

### This layer is a strided convolution
`patchify + project` is *exactly* a convolution with kernel size `P` and stride
`P`. Reshape your `(P²C, E)` weight to `(P, P, C, E)`, run
`conv_general_dilated(x, kernel, strides=(P, P), padding='VALID')` with
`dimension_numbers=('NHWC', 'HWIO', 'NHWC')`, and you get the same numbers —
one of the tests checks this. That is how the reference ViT implementation is
written, because a single conv kernel is fused and never materialises the
`(B, N, P²C)` intermediate. Knowing the equivalence tells you the "transformers
have no convolutions" claim is only true after layer one.

The equivalence also pins the memory story: the reshape route allocates
`B·N·P²C` floats — the same count as the image, just re-laid-out — so it is
cheap, but it does force a full transpose of the image.

### Why ViT needs a class token and position embeddings
A transformer is **permutation-equivariant**: shuffle the tokens and the outputs
shuffle with them. Patch order carries all the 2-D geometry, and after this
layer nothing in the model knows that patch 3 sits above patch 17. So ViT adds a
learned position embedding to every token right after this step. Sinusoids work
too, but learned embeddings are standard here, and they are why changing the
input resolution requires interpolating the position table.

The `[CLS]` token is a separate trick: you need one vector to classify from.
Mean-pooling the patch tokens works, but a prepended learned token gives
attention a free slot that starts with no spatial identity of its own and can
learn to aggregate whatever the head needs. Either way, this layer emits `N`
tokens and the model that wraps it feeds forward `N + 1`.

### The trap
`x.reshape(B, N, P*P*C)` straight from `(B, H, W, C)` produces the correct output
shape and completely wrong patches — it slices full image rows, not squares. The
shape assertion passes, the model trains, accuracy is mysteriously bad. Get the
transpose right.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class PatchEmbedding(nnx.Module):
    """Split an NHWC image into P x P patches and project each to embed_dim."""

    def __init__(
        self,
        img_size: int,
        patch_size: int,
        in_channels: int,
        embed_dim: int,
        *,
        rngs: nnx.Rngs,
    ):
        # Remember to set self.num_patches
        pass  # Replace this

    def __call__(self, x):
        """(B, H, W, C) -> (B, num_patches, embed_dim)"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

pe = PatchEmbedding(img_size=32, patch_size=8, in_channels=3, embed_dim=64,
                    rngs=nnx.Rngs(0))
x = jax.random.normal(jax.random.key(0), (2, 32, 32, 3))
print("image", x.shape, "->", pe(x).shape, " num_patches =", pe.num_patches)

# It is a stride-P convolution. Same weight, reshaped into an HWIO kernel.
P, C, E = 8, 3, 64
kernel = pe.w[...].reshape(P, P, C, E)
conv = jax.lax.conv_general_dilated(
    x, kernel, window_strides=(P, P), padding="VALID",
    dimension_numbers=("NHWC", "HWIO", "NHWC"),
)
conv = conv.reshape(x.shape[0], -1, E) + pe.b[...]
print("max |reshape - conv| =", float(jnp.abs(pe(x) - conv).max()))

# The trap: reshaping without the transpose gives full image ROWS, not squares.
naive = x.reshape(2, 16, 8 * 8 * 3)
correct = x.reshape(2, 4, 8, 4, 8, 3).transpose(0, 1, 3, 2, 4, 5).reshape(2, 16, 8 * 8 * 3)
print("naive reshape equals the real patches?", bool(jnp.allclose(naive, correct)))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("vit_patch")

# hint("vit_patch")      # stuck? nudge without the answer
# solution("vit_patch")  # spoiler: the reference implementation